# Notebook 03: Carga de Datos
## ETL Optimización de Rutas - TransCarga S.A.S.

**Objetivo**: Cargar datos procesados y generar archivos finales para el modelo

**Tiempo estimado**: 10-15 minutos

In [1]:
# CELDA 1: Configuración Inicial
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime
import logging
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configurar paths
BASE_DIR = r'C:\Users\danie\OneDrive\Documentos\TransCarga_ETL'
RAW_DIR = os.path.join(BASE_DIR, 'datos', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'datos', 'processed')
OUTPUT_DIR = os.path.join(BASE_DIR, 'datos', 'output')
LOGS_DIR = os.path.join(BASE_DIR, 'logs')

# Crear directorios
for d in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(os.path.join(LOGS_DIR, 'carga.log'), mode='a'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print(f"✓ Directorio processed: {PROCESSED_DIR}")
print(f"✓ Directorio output: {OUTPUT_DIR}")
logger.info("Environment de carga configurado")

2026-05-11 13:13:02,018 - INFO - Environment de carga configurado


✓ Directorio processed: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\processed
✓ Directorio output: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\output


In [2]:
# CELDA 2: Cargar datos procesados
def cargar_datos_procesados():
    """
    Carga todos los archivos procesados
    """
    datos = {}
    
    archivos = [
        ('clientes', 'clientes_clean.csv'),
        ('vehiculos', 'vehiculos_clean.csv'),
        ('trafico', 'trafico_clean.csv'),
        ('peajes', 'peajes_clean.csv'),
        ('combustible', 'combustible_clean.csv'),
        ('matriz_distancias', 'matriz_distancias.csv'),
        ('matriz_tiempos', 'matriz_tiempos.csv')
    ]
    
    for nombre, archivo in archivos:
        path = os.path.join(PROCESSED_DIR, archivo)
        if os.path.exists(path):
            if 'matriz' in archivo:
                datos[nombre] = pd.read_csv(path, index_col=0)
            else:
                datos[nombre] = pd.read_csv(path)
            logger.info(f"{nombre}: {datos[nombre].shape[0]} registros")
            print(f"✓ {nombre}: {datos[nombre].shape}")
        else:
            logger.warning(f"{nombre}: no encontrado")
            print(f"✗ {nombre}: no encontrado")
    
    return datos

datos = cargar_datos_procesados()
print(f"\n✓ Total datasets: {len(datos)}")

2026-05-11 13:13:02,053 - INFO - clientes: 200 registros


2026-05-11 13:13:02,055 - INFO - vehiculos: 85 registros


2026-05-11 13:13:02,058 - INFO - trafico: 1176 registros


2026-05-11 13:13:02,059 - INFO - peajes: 6 registros


2026-05-11 13:13:02,065 - INFO - combustible: 1881 registros


2026-05-11 13:13:02,066 - INFO - matriz_distancias: 52 registros


2026-05-11 13:13:02,069 - INFO - matriz_tiempos: 52 registros


✓ clientes: (200, 16)
✓ vehiculos: (85, 13)
✓ trafico: (1176, 10)
✓ peajes: (6, 11)
✓ combustible: (1881, 11)
✓ matriz_distancias: (52, 52)
✓ matriz_tiempos: (52, 52)

✓ Total datasets: 7


In [3]:
# CELDA 3: Crear dataset consolidado para optimización
def crear_dataset_optimizacion(datos):
    """
    Crea dataset consolidado para el modelo de optimización
    """
    # Clientes con coordenadas
    clientes = datos['clientes'][['id_cliente', 'nombre', 'municipio', 'latitud', 'longitud', 
                                   'capacidad_kg', 'apertura_minutos', 'cierre_minutos', 
                                   'prioridad', 'valor_pedido']].copy()
    
    # Vehículos disponibles
    vehiculos = datos['vehiculos'][datos['vehiculos']['disponible']==1].copy()
    vehiculos = vehiculos[['id_vehiculo', 'tipo', 'capacidad_kg', 'velocidad_promedio', 
                           'bodega_base', 'costo_hora']]
    
    # Crear bodegas
    bodegas = pd.DataFrame([
        {'id_bodega': 'B001', 'nombre': 'Bodega Medellin', 'ciudad': 'Medellin', 
         'latitud': 6.2518, 'longitud': -75.5636, 'capacidad_max': 10000},
        {'id_bodega': 'B002', 'nombre': 'Bodega Cali', 'ciudad': 'Cali', 
         'latitud': 3.4516, 'longitud': -76.5320, 'capacidad_max': 8000}
    ])
    
    # Precio combustible promedio
    precio_combustible = datos['combustible']['precio'].mean()
    
    # Guardar archivos
    clientes_path = os.path.join(OUTPUT_DIR, 'optimizacion_clientes.csv')
    vehiculos_path = os.path.join(OUTPUT_DIR, 'optimizacion_vehiculos.csv')
    bodegas_path = os.path.join(OUTPUT_DIR, 'optimizacion_bodegas.csv')
    
    clientes.to_csv(clientes_path, index=False)
    vehiculos.to_csv(vehiculos_path, index=False)
    bodegas.to_csv(bodegas_path, index=False)
    
    # Copiar matrices
    matriz_dist_path = os.path.join(OUTPUT_DIR, 'optimizacion_matriz_distancias.csv')
    matriz_tiempo_path = os.path.join(OUTPUT_DIR, 'optimizacion_matriz_tiempos.csv')
    
    datos['matriz_distancias'].to_csv(matriz_dist_path)
    datos['matriz_tiempos'].to_csv(matriz_tiempo_path)
    
    print(f"✓ Dataset de optimización creado:")
    print(f"  - Clientes: {len(clientes)}")
    print(f"  - Vehículos disponibles: {len(vehiculos)}")
    print(f"  - Bodegas: {len(bodegas)}")
    print(f"  - Precio combustible promedio: ${precio_combustible:,.0f}/galón")
    
    logger.info(f"Dataset optimización creado: {len(clientes)} clientes, {len(vehiculos)} vehículos")
    
    return clientes, vehiculos, bodegas

if 'clientes' in datos and 'vehiculos' in datos:
    clientes_opt, vehiculos_opt, bodegas_opt = crear_dataset_optimizacion(datos)

2026-05-11 13:13:02,090 - INFO - Dataset optimización creado: 200 clientes, 64 vehículos


✓ Dataset de optimización creado:
  - Clientes: 200
  - Vehículos disponibles: 64
  - Bodegas: 2
  - Precio combustible promedio: $9,294/galón


In [4]:
# CELDA 4: Crear archivo de configuración del modelo
def crear_config_modelo(datos):
    """
    Crea archivo de configuración JSON para el modelo VRP
    """
    config = {
        'proyecto': 'Optimizacion de Rutas TransCarga S.A.S.',
        'version': '1.0',
        'fecha_creacion': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        
        'parametros_vrp': {
            'tipo_modelo': 'CVRP_TW',  # Capacitated VRP with Time Windows
            'objetivo': 'minimizar_costo_total',
            'restricciones': [
                'capacidad_vehiculo',
                'ventana_tiempo_cliente',
                'horario_trabajo_conductor',
                'visita_unica'
            ]
        },
        
        'parametros_generales': {
            'hora_inicio_operaciones': 6,  # 6:00 AM
            'hora_fin_operaciones': 20,    # 8:00 PM
            'max_paradas_ruta': 15,
            'tiempo_servicio_minutos': 15
        },
        
        'costos': {
            'precio_combustible_galon': float(datos['combustible']['precio'].mean()) if 'combustible' in datos else 13000,
            'costo_mantenimiento_km': 50,
            'costo_conductor_hora': 25000,
            'penalizacion_tarde_minuto': 1000
        },
        
        'estadisticas': {
            'total_clientes': len(datos['clientes']) if 'clientes' in datos else 0,
            'total_vehiculos': len(datos['vehiculos']) if 'vehiculos' in datos else 0,
            'vehiculos_disponibles': int(datos['vehiculos']['disponible'].sum()) if 'vehiculos' in datos else 0
        }
    }
    
    config_path = os.path.join(OUTPUT_DIR, 'config_modelo.json')
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(config, f, indent=2, ensure_ascii=False)
    
    print(f"✓ Configuración del modelo guardada: {config_path}")
    print(f"\nConfiguración:")
    print(json.dumps(config, indent=2, ensure_ascii=False))
    
    logger.info("Configuración del modelo creada")
    
    return config

config = crear_config_modelo(datos)

2026-05-11 13:13:02,099 - INFO - Configuración del modelo creada


✓ Configuración del modelo guardada: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\output\config_modelo.json

Configuración:
{
  "proyecto": "Optimizacion de Rutas TransCarga S.A.S.",
  "version": "1.0",
  "fecha_creacion": "2026-05-11 13:13:02",
  "parametros_vrp": {
    "tipo_modelo": "CVRP_TW",
    "objetivo": "minimizar_costo_total",
    "restricciones": [
      "capacidad_vehiculo",
      "ventana_tiempo_cliente",
      "horario_trabajo_conductor",
      "visita_unica"
    ]
  },
  "parametros_generales": {
    "hora_inicio_operaciones": 6,
    "hora_fin_operaciones": 20,
    "max_paradas_ruta": 15,
    "tiempo_servicio_minutos": 15
  },
  "costos": {
    "precio_combustible_galon": 9293.692185007974,
    "costo_mantenimiento_km": 50,
    "costo_conductor_hora": 25000,
    "penalizacion_tarde_minuto": 1000
  },
  "estadisticas": {
    "total_clientes": 200,
    "total_vehiculos": 85,
    "vehiculos_disponibles": 64
  }
}


In [5]:
# CELDA 5: Generar estadísticas finales
def generar_estadisticas(datos):
    """
    Genera estadísticas descriptivas del dataset
    """
    stats = {}
    
    # Clientes
    if 'clientes' in datos:
        clientes = datos['clientes']
        stats['clientes'] = {
            'total': len(clientes),
            'medellin': len(clientes[clientes['municipio'].str.contains('MEDELL', na=False)]),
            'cali': len(clientes[clientes['municipio'].str.contains('CALI', na=False)]),
            'capacidad_promedio_kg': round(clientes['capacidad_kg'].mean(), 2),
            'valor_pedido_promedio': round(clientes['valor_pedido'].mean(), 0),
            'ventana_tiempo_promedio_horas': round(clientes['ventana_horas'].mean(), 1)
        }
    
    # Vehículos
    if 'vehiculos' in datos:
        vehiculos = datos['vehiculos']
        stats['vehiculos'] = {
            'total': len(vehiculos),
            'disponibles': int(vehiculos['disponible'].sum()),
            'capacidad_total_kg': int(vehiculos['capacidad_kg'].sum()),
            'tipos': vehiculos['tipo'].value_counts().to_dict()
        }
    
    # Matrices
    if 'matriz_distancias' in datos:
        matriz = datos['matriz_distancias'].values
        # Excluir diagonal
        mask = matriz > 0
        stats['distancias'] = {
            'puntos': matriz.shape[0],
            'min_km': round(matriz[mask].min(), 2),
            'max_km': round(matriz[mask].max(), 2),
            'promedio_km': round(matriz[mask].mean(), 2)
        }
    
    # Guardar estadísticas
    stats_path = os.path.join(OUTPUT_DIR, 'estadisticas_dataset.json')
    with open(stats_path, 'w', encoding='utf-8') as f:
        json.dump(stats, f, indent=2, ensure_ascii=False)
    
    print("=" * 60)
    print("ESTADÍSTICAS DEL DATASET")
    print("=" * 60)
    print(json.dumps(stats, indent=2, ensure_ascii=False))
    
    logger.info("Estadísticas generadas")
    
    return stats

stats = generar_estadisticas(datos)

2026-05-11 13:13:02,110 - INFO - Estadísticas generadas


ESTADÍSTICAS DEL DATASET
{
  "clientes": {
    "total": 200,
    "medellin": 1,
    "cali": 0,
    "capacidad_promedio_kg": 343.75,
    "valor_pedido_promedio": 2448013.0,
    "ventana_tiempo_promedio_horas": 11.0
  },
  "vehiculos": {
    "total": 85,
    "disponibles": 64,
    "capacidad_total_kg": 52000,
    "tipos": {
      "Automovil": 27,
      "Camioneta": 27,
      "Motocicleta": 12,
      "Camion": 11,
      "Bus": 8
    }
  },
  "distancias": {
    "puntos": 52,
    "min_km": 0.68,
    "max_km": 593.62,
    "promedio_km": 177.43
  }
}


In [6]:
# CELDA 6: Validar integridad de datos
def validar_datos(datos):
    """
    Valida la integridad de los datos procesados
    """
    print("=" * 60)
    print("VALIDACIÓN DE INTEGRIDAD")
    print("=" * 60)
    
    validaciones = []
    
    # Validar clientes
    if 'clientes' in datos:
        clientes = datos['clientes']
        nulos = clientes.isnull().sum().sum()
        duplicados = clientes.duplicated(subset=['id_cliente']).sum()
        
        validaciones.append({
            'Dataset': 'Clientes',
            'Registros': len(clientes),
            'Nulos': nulos,
            'Duplicados': duplicados,
            'Estado': '✓ OK' if nulos == 0 and duplicados == 0 else '⚠ Problema'
        })
    
    # Validar vehículos
    if 'vehiculos' in datos:
        vehiculos = datos['vehiculos']
        nulos = vehiculos.isnull().sum().sum()
        duplicados = vehiculos.duplicated(subset=['id_vehiculo']).sum()
        
        validaciones.append({
            'Dataset': 'Vehículos',
            'Registros': len(vehiculos),
            'Nulos': nulos,
            'Duplicados': duplicados,
            'Estado': '✓ OK' if nulos == 0 and duplicados == 0 else '⚠ Problema'
        })
    
    # Validar matrices
    for matriz_name in ['matriz_distancias', 'matriz_tiempos']:
        if matriz_name in datos:
            matriz = datos[matriz_name]
            nulos = matriz.isnull().sum().sum()
            simetrica = (matriz.values.T == matriz.values).all()
            
            validaciones.append({
                'Dataset': matriz_name.replace('_', ' ').title(),
                'Registros': f"{matriz.shape[0]}x{matriz.shape[1]}",
                'Nulos': nulos,
                'Duplicados': '-' if simetrica else '⚠ No simétrica',
                'Estado': '✓ OK' if nulos == 0 else '⚠ Problema'
            })
    
    df_validaciones = pd.DataFrame(validaciones)
    display(df_validaciones)
    
    # Guardar reporte
    validaciones_path = os.path.join(OUTPUT_DIR, 'reporte_validacion.csv')
    df_validaciones.to_csv(validaciones_path, index=False)
    
    logger.info("Validación completada")
    
    return df_validaciones

validaciones = validar_datos(datos)

VALIDACIÓN DE INTEGRIDAD


,Dataset,Registros,Nulos,Duplicados,Estado
0,Clientes,200,0,0,✓ OK
1,Vehículos,85,0,0,✓ OK
2,Matriz Distancias,52x52,0,-,✓ OK
3,Matriz Tiempos,52x52,0,-,✓ OK


2026-05-11 13:13:02,131 - INFO - Validación completada


In [7]:
# CELDA 7: Generar reporte final
def generar_reporte_final():
    """
    Genera reporte final del ETL
    """
    print("\n" + "=" * 80)
    print("REPORTE FINAL - ETL OPTIMIZACIÓN DE RUTAS")
    print("TransCarga S.A.S.")
    print("=" * 80)
    
    # Listar archivos generados
    print("\n📁 ARCHIVOS GENERADOS EN OUTPUT:")
    print("-" * 40)
    
    for archivo in os.listdir(OUTPUT_DIR):
        path = os.path.join(OUTPUT_DIR, archivo)
        size = os.path.getsize(path) / 1024
        print(f"  ✓ {archivo} ({size:.1f} KB)")
    
    print("\n📊 RESUMEN:")
    print("-" * 40)
    print(f"  • Clientes procesados: {len(datos.get('clientes', []))}")
    print(f"  • Vehículos disponibles: {int(datos['vehiculos']['disponible'].sum()) if 'vehiculos' in datos else 0}")
    print(f"  • Bodegas: 2")
    print(f"  • Matriz de distancias: {datos['matriz_distancias'].shape if 'matriz_distancias' in datos else 'N/A'}")
    
    print("\n✅ ETL COMPLETADO EXITOSAMENTE")
    print(f"   Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Output: {OUTPUT_DIR}")
    
    logger.info("ETL completado exitosamente")

generar_reporte_final()

2026-05-11 13:13:02,138 - INFO - ETL completado exitosamente



REPORTE FINAL - ETL OPTIMIZACIÓN DE RUTAS
TransCarga S.A.S.

📁 ARCHIVOS GENERADOS EN OUTPUT:
----------------------------------------
  ✓ config_modelo.json (0.8 KB)
  ✓ estadisticas_dataset.json (0.5 KB)
  ✓ optimizacion_bodegas.csv (0.1 KB)
  ✓ optimizacion_clientes.csv (21.6 KB)
  ✓ optimizacion_matriz_distancias.csv (18.0 KB)
  ✓ optimizacion_matriz_tiempos.csv (16.3 KB)
  ✓ optimizacion_vehiculos.csv (2.3 KB)
  ✓ reporte_validacion.csv (0.2 KB)

📊 RESUMEN:
----------------------------------------
  • Clientes procesados: 200
  • Vehículos disponibles: 64
  • Bodegas: 2
  • Matriz de distancias: (52, 52)

✅ ETL COMPLETADO EXITOSAMENTE
   Fecha: 2026-05-11 13:13:02
   Output: C:\Users\danie\OneDrive\Documentos\TransCarga_ETL\datos\output


---
## ✅ ETL Completado

Los datos están listos para ser utilizados en el modelo de optimización de rutas.

### Próximos Pasos:
1. Ejecutar el modelo de optimización (Google OR-Tools)
2. Validar resultados con datos reales
3. Ajustar parámetros según necesidades operativas